In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

In [14]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
MODELS_DIR = PROJECT_ROOT / "models"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW:", DATA_RAW)
print("MODELS_DIR:", MODELS_DIR)

PROJECT_ROOT: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand
DATA_RAW: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\data\raw
MODELS_DIR: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\models


In [15]:
train = pd.read_csv(DATA_RAW / "train.csv")

print("train shape:", train.shape)
display(train.head())

train shape: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [16]:
TARGET_COL = "count"

LEAKAGE_COLS = [
    "casual",
    "registered",
]

RAW_DATETIME_COL = "datetime"

EXCLUDED_COLS = [
    TARGET_COL,
    "casual",
    "registered",
    RAW_DATETIME_COL,
    "day",
]

FINAL_FEATURES = [
    "season",
    "holiday",
    "workingday",
    "weather",
    "temp",
    "atemp",
    "humidity",
    "windspeed",
    "year",
    "month",
    "hour",
    "weekday",
    "is_weekend",
]

FROZEN_MODEL_PARAMS = {
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.1,
    "random_state": 42,
}

VALIDATION_METRICS = {
    "RMSLE": 0.306045,
    "RMSE": 47.065693,
    "MAE": 28.274786,
}

VALIDATION_SPLIT_DESCRIPTION = {
    "local_train": "days 1-15 of each month from train.csv",
    "local_validation": "days 16-19 of each month from train.csv",
    "note": (
        "These are final local validation metrics, not independent final test metrics. "
        "Validation days 16-19 were used during model selection and limited tuning."
    ),
}

print("Final features:", FINAL_FEATURES)
print("Excluded columns:", EXCLUDED_COLS)
print("Frozen params:", FROZEN_MODEL_PARAMS)

Final features: ['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend']
Excluded columns: ['count', 'casual', 'registered', 'datetime', 'day']
Frozen params: {'max_iter': 200, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.1, 'random_state': 42}


In [17]:
class BikeDatetimeFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Create datetime-derived features and return the final model feature matrix.

    Expected raw input columns:
    - datetime
    - season
    - holiday
    - workingday
    - weather
    - temp
    - atemp
    - humidity
    - windspeed

    Optional columns that will be ignored if present:
    - count
    - casual
    - registered
    - day
    """

    def __init__(self, datetime_col="datetime", final_features=None):
        self.datetime_col = datetime_col
        self.final_features = final_features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_transformed = X.copy()

        if self.datetime_col not in X_transformed.columns:
            raise ValueError(f"Missing required datetime column: {self.datetime_col}")

        X_transformed[self.datetime_col] = pd.to_datetime(X_transformed[self.datetime_col])

        X_transformed["year"] = X_transformed[self.datetime_col].dt.year
        X_transformed["month"] = X_transformed[self.datetime_col].dt.month
        X_transformed["hour"] = X_transformed[self.datetime_col].dt.hour
        X_transformed["weekday"] = X_transformed[self.datetime_col].dt.weekday
        X_transformed["is_weekend"] = X_transformed["weekday"].isin([5, 6]).astype(int)

        missing_features = [
            col for col in self.final_features
            if col not in X_transformed.columns
        ]

        if missing_features:
            raise ValueError(f"Missing required feature columns after transform: {missing_features}")

        return X_transformed[self.final_features].copy()

In [18]:
X_raw = train.drop(columns=[TARGET_COL]).copy()
y = train[TARGET_COL].copy()

print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)

print("X_raw columns:")
print(X_raw.columns.tolist())

X_raw shape: (10886, 11)
y shape: (10886,)
X_raw columns:
['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'casual', 'registered']


In [19]:
X_raw = train.drop(columns=[TARGET_COL, "casual", "registered"]).copy()
y = train[TARGET_COL].copy()

print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)

print("Target in X_raw:", TARGET_COL in X_raw.columns)
print("Leakage columns in X_raw:", set(X_raw.columns).intersection(LEAKAGE_COLS))

X_raw shape: (10886, 9)
y shape: (10886,)
Target in X_raw: False
Leakage columns in X_raw: set()


In [20]:
base_model = HistGradientBoostingRegressor(**FROZEN_MODEL_PARAMS)

regressor = TransformedTargetRegressor(
    regressor=base_model,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

final_pipeline = Pipeline(
    steps=[
        ("datetime_features", BikeDatetimeFeatureEngineer(
            datetime_col=RAW_DATETIME_COL,
            final_features=FINAL_FEATURES,
        )),
        ("model", regressor),
    ]
)

final_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('datetime_features', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,datetime_col,'datetime'
,final_features,"['season', 'holiday', ...]"
,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",HistGradientB...ndom_state=42)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",False


In [21]:
train_for_split = train.copy()
train_for_split["datetime"] = pd.to_datetime(train_for_split["datetime"])
train_for_split["day"] = train_for_split["datetime"].dt.day

local_train_mask = train_for_split["day"] <= 15
local_valid_mask = train_for_split["day"].between(16, 19)

X_train_raw = train.loc[local_train_mask].drop(columns=[TARGET_COL, "casual", "registered"]).copy()
y_train = train.loc[local_train_mask, TARGET_COL].copy()

X_valid_raw = train.loc[local_valid_mask].drop(columns=[TARGET_COL, "casual", "registered"]).copy()
y_valid = train.loc[local_valid_mask, TARGET_COL].copy()

print("X_train_raw shape:", X_train_raw.shape)
print("y_train shape:", y_train.shape)
print("X_valid_raw shape:", X_valid_raw.shape)
print("y_valid shape:", y_valid.shape)

X_train_raw shape: (8600, 9)
y_train shape: (8600,)
X_valid_raw shape: (2286, 9)
y_valid shape: (2286,)


In [22]:
def clipped_predictions(y_pred):
    return np.clip(y_pred, a_min=0, a_max=None)


def regression_metrics(y_true, y_pred):
    y_pred_clipped = clipped_predictions(y_pred)

    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred_clipped))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "min_pred_raw": np.min(y_pred),
        "min_pred_clipped": np.min(y_pred_clipped),
        "max_pred_raw": np.max(y_pred),
    }

In [24]:
validation_pipeline = Pipeline(
    steps=[
        ("datetime_features", BikeDatetimeFeatureEngineer(
            datetime_col=RAW_DATETIME_COL,
            final_features=FINAL_FEATURES,
        )),
        ("model", TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(**FROZEN_MODEL_PARAMS),
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False,
        )),
    ]
)

validation_pipeline.fit(X_train_raw, y_train)
y_valid_pred = validation_pipeline.predict(X_valid_raw)

validation_pipeline_metrics = regression_metrics(y_valid, y_valid_pred)

validation_pipeline_metrics_df = pd.DataFrame(
    [validation_pipeline_metrics],
    index=["validation_pipeline"]
)

validation_pipeline_metrics_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw
validation_pipeline,0.306045,47.065693,28.274786,1.138463,1.138463,825.95043


In [25]:
# Strict raw feature matrix for final artifact training:
# exclude target and target components before passing data to the pipeline.
X_final_raw = train.drop(columns=[TARGET_COL, "casual", "registered"]).copy()
y_final = train[TARGET_COL].copy()

print("X_final_raw shape:", X_final_raw.shape)
print("y_final shape:", y_final.shape)

print("Target in X_final_raw:", TARGET_COL in X_final_raw.columns)
print("Leakage columns in X_final_raw:", set(X_final_raw.columns).intersection(LEAKAGE_COLS))

final_pipeline.fit(X_final_raw, y_final)

print("Final pipeline fitted on full train.csv after candidate freeze.")

X_final_raw shape: (10886, 9)
y_final shape: (10886,)
Target in X_final_raw: False
Leakage columns in X_final_raw: set()
Final pipeline fitted on full train.csv after candidate freeze.


In [26]:
import joblib
from datetime import datetime, timezone

In [27]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

bundle = {
    "pipeline": final_pipeline,
    "target_policy": {
        "strategy": "log1p",
        "fit_transform": "np.log1p(y)",
        "inverse_transform": "np.expm1(pred_log)",
        "prediction_postprocessing": "clip predictions at 0 for RMSLE safety if needed",
    },
    "model": {
        "name": "HistGradientBoostingRegressor",
        "params": FROZEN_MODEL_PARAMS,
    },
    "feature_engineering": {
        "transformer": "BikeDatetimeFeatureEngineer",
        "datetime_col": RAW_DATETIME_COL,
        "derived_features": [
            "year",
            "month",
            "hour",
            "weekday",
            "is_weekend",
        ],
        "final_features": FINAL_FEATURES,
        "excluded_columns": EXCLUDED_COLS,
        "day_policy": "excluded from main feature set due to transfer risk",
    },
    "validation": {
        "metrics": VALIDATION_METRICS,
        "split_description": VALIDATION_SPLIT_DESCRIPTION,
        "important_note": (
            "These are final local validation metrics, not independent final test metrics. "
            "Validation days 16-19 were used during model selection and limited tuning."
        ),
    },
    "training_policy": {
        "artifact_fit_data": "full labeled train.csv after candidate freeze",
        "artifact_fit_rows": len(X_final_raw),
        "no_official_test_used": True,
        "no_kaggle_submission_created": True,
        "no_final_independent_test_available": True,
    },
    "metadata": {
        "project": "05_bike_sharing_demand",
        "stage": "Stage 7 - Save final pipeline and update README",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    },
}

artifact_path = MODELS_DIR / "bike_sharing_final_bundle.joblib"

joblib.dump(bundle, artifact_path)

artifact_path

WindowsPath('c:/temp/python_learning/ml_projects/ml_projects_batch_01/05_bike_sharing_demand/models/bike_sharing_final_bundle.joblib')

In [28]:
print("Artifact path:", artifact_path)
print("Artifact exists:", artifact_path.exists())
print("Artifact size MB:", artifact_path.stat().st_size / (1024 * 1024))

Artifact path: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\models\bike_sharing_final_bundle.joblib
Artifact exists: True
Artifact size MB: 0.7012033462524414


In [31]:
loaded_bundle = joblib.load(artifact_path)

print("Loaded bundle keys:")
print(loaded_bundle.keys())

loaded_pipeline = loaded_bundle["pipeline"]

print("\nLoaded model:")
print(loaded_bundle["model"])

print("\nLoaded target policy:")
print(loaded_bundle["target_policy"])

Loaded bundle keys:
dict_keys(['pipeline', 'target_policy', 'model', 'feature_engineering', 'validation', 'training_policy', 'metadata'])

Loaded model:
{'name': 'HistGradientBoostingRegressor', 'params': {'max_iter': 200, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.1, 'random_state': 42}}

Loaded target policy:
{'strategy': 'log1p', 'fit_transform': 'np.log1p(y)', 'inverse_transform': 'np.expm1(pred_log)', 'prediction_postprocessing': 'clip predictions at 0 for RMSLE safety if needed'}


In [32]:
sample_raw = X_final_raw.head(10).copy()

sample_predictions = loaded_pipeline.predict(sample_raw)
sample_predictions_clipped = np.clip(sample_predictions, a_min=0, a_max=None)

sample_prediction_df = sample_raw.copy()
sample_prediction_df["prediction_raw"] = sample_predictions
sample_prediction_df["prediction_clipped"] = sample_predictions_clipped

display(sample_prediction_df)

print("Min raw prediction:", sample_predictions.min())
print("Min clipped prediction:", sample_predictions_clipped.min())
print("All clipped predictions non-negative:", (sample_predictions_clipped >= 0).all())

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,prediction_raw,prediction_clipped
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,27.446979,27.446979
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,26.245127,26.245127
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,13.851522,13.851522
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,8.817132,8.817132
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,2.738271,2.738271
5,2011-01-01 05:00:00,1,0,0,2,9.84,12.880,75,6.0032,2.253247,2.253247
6,2011-01-01 06:00:00,1,0,0,1,9.02,13.635,80,0.0000,2.269582,2.269582
7,2011-01-01 07:00:00,1,0,0,1,8.20,12.880,86,0.0000,6.319202,6.319202
8,2011-01-01 08:00:00,1,0,0,1,9.84,14.395,75,0.0000,16.122727,16.122727
9,2011-01-01 09:00:00,1,0,0,1,13.12,17.425,76,0.0000,32.322688,32.322688


Min raw prediction: 2.253246740041657
Min clipped prediction: 2.253246740041657
All clipped predictions non-negative: True


In [33]:
sample_raw = X_final_raw.head(10).copy()

sample_predictions = loaded_pipeline.predict(sample_raw)
sample_predictions_clipped = np.clip(sample_predictions, a_min=0, a_max=None)

sample_prediction_df = sample_raw.copy()
sample_prediction_df["prediction_raw"] = sample_predictions
sample_prediction_df["prediction_clipped"] = sample_predictions_clipped

display(sample_prediction_df)

print("Min raw prediction:", sample_predictions.min())
print("Min clipped prediction:", sample_predictions_clipped.min())
print("All clipped predictions non-negative:", (sample_predictions_clipped >= 0).all())

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,prediction_raw,prediction_clipped
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,27.446979,27.446979
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,26.245127,26.245127
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,13.851522,13.851522
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,8.817132,8.817132
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,2.738271,2.738271
5,2011-01-01 05:00:00,1,0,0,2,9.84,12.880,75,6.0032,2.253247,2.253247
6,2011-01-01 06:00:00,1,0,0,1,9.02,13.635,80,0.0000,2.269582,2.269582
7,2011-01-01 07:00:00,1,0,0,1,8.20,12.880,86,0.0000,6.319202,6.319202
8,2011-01-01 08:00:00,1,0,0,1,9.84,14.395,75,0.0000,16.122727,16.122727
9,2011-01-01 09:00:00,1,0,0,1,13.12,17.425,76,0.0000,32.322688,32.322688


Min raw prediction: 2.253246740041657
Min clipped prediction: 2.253246740041657
All clipped predictions non-negative: True


In [34]:
inference_like_sample = train[
    [
        "datetime",
        "season",
        "holiday",
        "workingday",
        "weather",
        "temp",
        "atemp",
        "humidity",
        "windspeed",
    ]
].head(10).copy()

inference_like_predictions = loaded_pipeline.predict(inference_like_sample)
inference_like_predictions_clipped = np.clip(
    inference_like_predictions,
    a_min=0,
    a_max=None,
)

print("Inference-like sample shape:", inference_like_sample.shape)
print("Predictions shape:", inference_like_predictions.shape)
print("Min prediction:", inference_like_predictions.min())
print("All clipped predictions non-negative:", (inference_like_predictions_clipped >= 0).all())

display(inference_like_sample.assign(prediction=inference_like_predictions_clipped))

Inference-like sample shape: (10, 9)
Predictions shape: (10,)
Min prediction: 2.253246740041657
All clipped predictions non-negative: True


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,prediction
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,27.446979
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,26.245127
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,13.851522
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,8.817132
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,2.738271
5,2011-01-01 05:00:00,1,0,0,2,9.84,12.880,75,6.0032,2.253247
6,2011-01-01 06:00:00,1,0,0,1,9.02,13.635,80,0.0000,2.269582
7,2011-01-01 07:00:00,1,0,0,1,8.20,12.880,86,0.0000,6.319202
8,2011-01-01 08:00:00,1,0,0,1,9.84,14.395,75,0.0000,16.122727
9,2011-01-01 09:00:00,1,0,0,1,13.12,17.425,76,0.0000,32.322688


## Final artifact notes

The final artifact was fitted on all available labeled `train.csv` rows after the candidate was frozen.

Important:

- This does not create a new validation or test score.
- Reported metrics remain the final local validation metrics from days 16–19.
- The official Kaggle `test.csv` was not used.
- No Kaggle submission was created.
- The artifact is saved under `models/`, which is ignored by Git.

Saved artifact:

- `models/bike_sharing_final_bundle.joblib`

The bundle includes:

- fitted pipeline
- datetime feature engineering transformer
- `HistGradientBoostingRegressor`
- `log1p` / `expm1` target policy metadata
- final feature list
- excluded columns
- validation metrics
- validation split description
- model params
- training policy metadata